In [ ]:
import csv
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
import simplekml
import random

file_path = "output.csv"

In [ ]:
lats = []
lons = []

with open(file_path, newline='') as f:
    reader = csv.DictReader(f)
    for row in reader:
        if row["hour"] == "1774461600":
            if row["lat"] and row["lon"]:
                lats.append(float(row["lat"]))
                lons.append(float(row["lon"]))

lats = np.array(lats)
lons = np.array(lons)
coords = np.stack([lats, lons], axis=1)

lat_mean = np.mean(lats)
lat_std = np.std(lats)
lon_mean = np.mean(lons)
lon_std = np.std(lons)

coords_norm = np.stack([
    (lats - lat_mean) / lat_std,
    (lons - lon_mean) / lon_std
], axis=1)

In [ ]:
SEQ_LEN = 100

def resample_path(path, target_len):
    idx = np.linspace(0, len(path) - 1, target_len).astype(int)
    return path[idx]

path = resample_path(coords_norm, SEQ_LEN)

start = path[0]
end = path[-1]

X = np.array([[start[0], start[1], end[0], end[1]]])  # shape (1, 4)
y = np.array([path])  # shape (1, SEQ_LEN, 2)

In [ ]:
def build_model(seq_len):
    inputs = layers.Input(shape=(4,))  # start + end

    x = layers.Dense(128, activation="relu")(inputs)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dense(seq_len * 2)(x)

    outputs = layers.Reshape((seq_len, 2))(x)

    model = models.Model(inputs, outputs)
    model.compile(optimizer="adam", loss="mse")

    return model

In [ ]:
model = build_model(SEQ_LEN)
model.fit(X, y, epochs=200, verbose=1)

In [ ]:
predicted = model.predict(X)[0]

In [ ]:
def denorm(path):
    lat = path[:,0] * lat_std + lat_mean
    lon = path[:,1] * lon_std + lon_mean
    return np.stack([lat, lon], axis=1)

real_path = denorm(path)
predicted_real = denorm(predicted)

In [ ]:
kml = simplekml.Kml()

# Visible (original)
visible_coords = [(lon, lat) for lat, lon in real_path]
line_visible = kml.newlinestring(name="Original Flight Path", coords=visible_coords)
line_visible.style.linestyle.color = simplekml.Color.green

# Predicted (reconstructed)
predicted_coords = [(lon, lat) for lat, lon in predicted_real]
line_pred = kml.newlinestring(name="Reconstructed Flight Path", coords=predicted_coords)
line_pred.style.linestyle.color = simplekml.Color.red

# Marker at first coordinate
first_lat, first_lon = real_path[0]
predicted_first_lat, predicted_first_lon = predicted_real[0]
point = kml.newpoint(name="Real Start", coords=[(first_lon, first_lat)])
point = kml.newpoint(name="Predicted Start", coords=[(predicted_first_lon, predicted_first_lat)])

kml.save("flight_prediction.kml")